# Automotive RAG Question Answering System - Phase 1

This notebook demonstrates the end-to-end Phase 1 document processing pipeline. We will execute the following steps:
1. Document Ingestion (PDF, CSV, DOCX, XLSX, TXT, Images)
2. Cleaning Pipeline
3. Native Python Chunking
4. Metadata Generation
5. Embeddings creation (using sentence-transformers)
6. FAISS Vector Store Indexing
7. Top-K Retrieval

In [ ]:
# Install requirements
!pip install -r ../requirements.txt

In [ ]:
import sys
import os
# Ensure src modules can be imported
sys.path.append(os.path.abspath('..'))

from src.processing import process_document
from src.chunking import recursive_chunking
from src.embeddings import generate_embeddings
from src.vector_store import VectorStore
from src.retriever import Retriever

### Step 1-4: Ingestion, Cleaning, Metadata Generation & Chunking

In [ ]:
file_path = '../test_assets/sample.pdf'

# Process document extracts text, cleans it, and infers metadata
text, metadata = process_document(file_path)

# Apply Recursive Chunking
chunks = recursive_chunking(text, chunk_size=500)
metadata['num_chunks'] = len(chunks)
metadatas = [metadata.copy() for _ in chunks]

print(f"Extracted {len(chunks)} chunks from {file_path}.")
print("Sample Metadata:", metadata)

### Step 5-6: Embeddings & FAISS Indexing

In [ ]:
# Generate embeddings (stored as numpy arrays on CPU)
embeddings = generate_embeddings(chunks)
print(f"Embeddings shape: {embeddings.shape}")

# Initialize Vector Store and build index
vs = VectorStore()
vs.build_index(embeddings, chunks, metadatas)

# Save index to disk
index_dir = '../faiss_index'
vs.save_index(index_dir)

### Step 7: Top-K Retrieval

In [ ]:
# Load the index back from disk
vs_loaded = VectorStore()
vs_loaded.load_index(index_dir)

# Initialize Retriever and search
retriever = Retriever(vs_loaded)
query = "Test PDF content"
results = retriever.retrieve(query, top_k=1)

print("\n--- Top-K Results ---")
for r in results:
    print(f"Distance: {r['distance']:.4f}")
    print(f"Metadata: {r['metadata']}")
    print(f"Chunk: {r['chunk']}\n")